# LAB 1: Criacao de Indice Hibrido no OpenSearch
## Busca Hibrida com kNN + BM25 para Corpus Juridico Brasileiro

**Objetivo:** Criar um indice hibrido no OpenSearch combinando busca vetorial (kNN com embeddings) e busca lexical (BM25) para um corpus de documentos juridicos brasileiros.

**Publico-alvo:** MBA em Direito e Seguranca Publica

**Duracao estimada:** 60 minutos

## 1. Instalacao de Dependencias

In [ ]:
!pip install -q opensearch-py==2.3.1
!pip install -q sentence-transformers==2.2.2
!pip install -q torch==2.0.1
!pip install -q tqdm==4.66.1

print("Dependencias instaladas com sucesso!")

## 2. Configuracao do Ambiente

In [ ]:
import json
import os
from opensearchpy import OpenSearch
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore', message='Unverified HTTPS request')

print("Bibliotecas importadas com sucesso!")

In [ ]:
OPENSEARCH_HOST = os.getenv('OPENSEARCH_HOST', 'localhost')
OPENSEARCH_PORT = int(os.getenv('OPENSEARCH_PORT', 9200))
OPENSEARCH_USER = os.getenv('OPENSEARCH_USER', 'admin')
OPENSEARCH_PASSWORD = os.getenv('OPENSEARCH_PASSWORD', 'admin')
OPENSEARCH_VERIFY_CERTS = os.getenv('OPENSEARCH_VERIFY_CERTS', 'False').lower() == 'true'

try:
    client = OpenSearch(
        hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
        http_auth=(OPENSEARCH_USER, OPENSEARCH_PASSWORD),
        use_ssl=True,
        verify_certs=OPENSEARCH_VERIFY_CERTS,
        ssl_show_warn=False,
        http_compress=True
    )
    
    info = client.info()
    print(f"Conectado ao OpenSearch {info['version']['number']}")
except Exception as e:
    print(f"Erro ao conectar: {str(e)}")

## 3. Teoria: Indices Hibridos

Busca hibrida combina dois tipos fundamentais:

1. **Busca Vetorial (kNN)**: Baseada em embeddings semanticos
2. **Busca Lexical (BM25)**: Baseada em palavras-chave exatas

Mapeamento: knn_vector (1024 dims, HNSW, cosinesimil) + text com analyzer portugues

## 4. Criacao do Indice Hibrido

In [ ]:
INDEX_NAME = "corpus_juridico_aula4"

index_body = {
    "settings": {
        "index": {
            "knn": True,
            "knn.algo_param.ef_construction": 128,
            "knn.algo_param.m": 24,
            "number_of_shards": 1,
            "number_of_replicas": 0,
            "analysis": {
                "analyzer": {
                    "portuguese_analyzer": {
                        "type": "standard",
                        "stopwords": "_portuguese_"
                    }
                }
            }
        }
    },
    "mappings": {
        "properties": {
            "id": {"type": "keyword"},
            "knn_vector": {
                "type": "knn_vector",
                "dimension": 1024,
                "method": {
                    "name": "hnsw",
                    "space_type": "cosinesimil",
                    "engine": "faiss"
                }
            },
            "text": {
                "type": "text",
                "analyzer": "portuguese_analyzer"
            },
            "titulo": {
                "type": "text",
                "analyzer": "portuguese_analyzer"
            },
            "tipo_documento": {"type": "keyword"},
            "data_publicacao": {"type": "date", "format": "yyyy-MM-dd"},
            "timestamp": {"type": "date"}
        }
    }
}

try:
    client.indices.delete(index=INDEX_NAME)
    print(f"Indice anterior removido")
except:
    pass

try:
    response = client.indices.create(index=INDEX_NAME, body=index_body)
    print(f"Indice criado com sucesso!")
except Exception as e:
    print(f"Erro ao criar indice: {str(e)}")

## 5. Carregamento do Modelo de Embeddings

In [ ]:
MODEL_NAME = "BAAI/bge-m3"
EMBEDDING_DIMENSION = 1024

print(f"Carregando modelo {MODEL_NAME}...")

try:
    model = SentenceTransformer(MODEL_NAME, device='cpu')
    print(f"Modelo carregado! Dimensao: {EMBEDDING_DIMENSION}")
except Exception as e:
    print(f"Erro ao carregar modelo: {str(e)}")

## 6. Carregamento do Corpus Juridico

In [ ]:
corpus_data = [
    {
        "id": "doc001",
        "titulo": "Prisao Preventiva - Requisitos e Pressupostos",
        "text": "A prisao preventiva eh medida cautelar. Pressupostos: prova da existencia de crime e indicio suficiente de autoria. Cabivel quando necessaria para garantir ordem publica.",
        "tipo_documento": "Lei",
        "data_publicacao": "2023-01-15"
    },
    {
        "id": "doc002",
        "titulo": "Lei Maria da Penha - Protecao contra Violencia Domestica",
        "text": "Lei 11.340/2006 estabelece mecanismos para coibir violencia domestica. Define formas: fisica, psicologica, sexual, patrimonial e moral. Estabelece medidas protetivas.",
        "tipo_documento": "Lei",
        "data_publicacao": "2023-02-10"
    },
    {
        "id": "doc003",
        "titulo": "Codigo de Processo Penal",
        "text": "CPP regula relacoes processuais penais. Estabelece procedimentos para investigacao, acusacao e julgamento. Garante direitos fundamentais incluindo presuncao de inocencia.",
        "tipo_documento": "Lei",
        "data_publicacao": "2023-03-05"
    },
    {
        "id": "doc004",
        "titulo": "Habeas Corpus - Remedio Constitucional",
        "text": "Habeas corpus protege contra ilegalidade ou abuso de poder que resulte em privacao de liberdade. Pode ser preventivo ou repressivo. Qualquer pessoa pode impetrar.",
        "tipo_documento": "Jurisprudencia",
        "data_publicacao": "2023-04-12"
    },
    {
        "id": "doc005",
        "titulo": "Constitucionalismo e Direitos Fundamentais",
        "text": "Constituicao Federal 1988 estabelece direitos fundamentais. Inclui direitos individuais, coletivos, sociais. Destaca dignidade da pessoa humana como fundamento.",
        "tipo_documento": "Doutrina",
        "data_publicacao": "2023-05-01"
    },
    {
        "id": "doc006",
        "titulo": "Crimes contra Patrimonio",
        "text": "Roubo eh crime com emprego de forca ou ameaca. Furto eh subtracao simples sem violencia. Ambos sao crimes graves com penas variaveis.",
        "tipo_documento": "Lei",
        "data_publicacao": "2023-06-08"
    },
    {
        "id": "doc007",
        "titulo": "Lei Geral de Protecao de Dados - LGPD",
        "text": "Lei 13.709/2018 regula tratamento de dados pessoais. Estabelece principios: necessidade, finalidade, transparencia. Define direitos do titular.",
        "tipo_documento": "Lei",
        "data_publicacao": "2023-07-15"
    },
    {
        "id": "doc008",
        "titulo": "Seguranca Publica e Direitos Humanos",
        "text": "Seguranca publica eh responsabilidade estatal. Deve respeitar direitos humanos fundamentais. Inclui prevencao, repressao ao crime e investigacao criminal.",
        "tipo_documento": "Doutrina",
        "data_publicacao": "2023-08-22"
    },
    {
        "id": "doc009",
        "titulo": "Mandado de Seguranca",
        "text": "Mandado de seguranca protege direito liquido e certo contra ato ilegal ou abusivo. Cabe contra atos que violem norma legal. Nao cabe contra atos criminais.",
        "tipo_documento": "Jurisprudencia",
        "data_publicacao": "2023-09-10"
    },
    {
        "id": "doc010",
        "titulo": "Inquerito Policial",
        "text": "Inquerito policial eh procedimento investigatorio realizado por delegado. Destina-se a reunir elementos de prova e apurar autoria. Nao constitui ato judicial.",
        "tipo_documento": "Lei",
        "data_publicacao": "2023-10-05"
    },
    {
        "id": "doc011",
        "titulo": "Consentimento e Capacidade Juridica",
        "text": "Consentimento eh manifestacao de vontade conforme exigencia legal. Capacidade juridica eh aptidao para adquirir direitos e obrigacoes. Lei estabelece incapacidade relativa e absoluta.",
        "tipo_documento": "Doutrina",
        "data_publicacao": "2023-11-12"
    },
    {
        "id": "doc012",
        "titulo": "Responsabilidade Civil",
        "text": "Responsabilidade civil eh obrigacao de reparar dano causado a terceiro. Fundamenta-se em ilicito e nexo causal. Indenizacao busca reparar integralmente prejuizo.",
        "tipo_documento": "Lei",
        "data_publicacao": "2023-12-01"
    },
    {
        "id": "doc013",
        "titulo": "Crime de Trafico de Drogas",
        "text": "Lei 11.343/2006 tipifica trafico de drogas. Crime contra saude publica. Caracteriza-se por importar, exportar, produzir, distribuir droga ilegal.",
        "tipo_documento": "Lei",
        "data_publicacao": "2024-01-10"
    },
    {
        "id": "doc014",
        "titulo": "Principio do Devido Processo Legal",
        "text": "Devido processo legal garante procedimentos legais antes de privacao de liberdade. Inclui presuncao de inocencia e direito de defesa. Corolario do Estado Democratico.",
        "tipo_documento": "Doutrina",
        "data_publicacao": "2024-02-08"
    },
    {
        "id": "doc015",
        "titulo": "Investigacao Criminal Moderna",
        "text": "Investigacao criminal usa tecnicas forenses, analise de DNA e vigilancia. Deve observar legalidade e respeito a direitos. Evidencia ilicita eh inadmissivel.",
        "tipo_documento": "Jurisprudencia",
        "data_publicacao": "2024-03-15"
    },
    {
        "id": "doc016",
        "titulo": "Ato Administrativo",
        "text": "Ato administrativo eh manifestacao de vontade de agente administrativo. Caracteristicas: tipicidade, legalidade, imperatividade. Sujeita-se a controle jurisdicional.",
        "tipo_documento": "Doutrina",
        "data_publicacao": "2024-04-02"
    },
    {
        "id": "doc017",
        "titulo": "Justica Restaurativa",
        "text": "Justica restaurativa busca reparacao e reconciliacao entre vitima, infrator e comunidade. Utiliza mediacao e conferencias. Reconhecida legislacao brasileira.",
        "tipo_documento": "Doutrina",
        "data_publicacao": "2024-05-10"
    },
    {
        "id": "doc018",
        "titulo": "Crimes contra Administracao Publica",
        "text": "Crimes incluem corrupcao, peculato e concussao. Praticados por servidores publicos contra orgaos estatais. Punidos com severidade por lesionar interesse publico.",
        "tipo_documento": "Lei",
        "data_publicacao": "2024-06-08"
    },
    {
        "id": "doc019",
        "titulo": "Execucao Penal e Direitos do Preso",
        "text": "Execucao penal inicia com transito em julgado da condenacao. Lei 7.210/1984 estabelece direitos dos presos. Preso nao perde direitos nao inerentes a pena.",
        "tipo_documento": "Lei",
        "data_publicacao": "2024-07-20"
    },
    {
        "id": "doc020",
        "titulo": "Principio da Legalidade",
        "text": "Legalidade vincula autoridades de seguranca publica ao ordenamento juridico. Nenhuma atividade policial valida sem previsao em lei. Garantia contra arbitrariedade.",
        "tipo_documento": "Doutrina",
        "data_publicacao": "2024-08-15"
    }
]

print(f"Corpus carregado com {len(corpus_data)} documentos")

## 7. Geracao de Embeddings

In [ ]:
BATCH_SIZE = 8

texts_to_embed = [doc['text'] for doc in corpus_data]

print(f"Gerando embeddings para {len(texts_to_embed)} documentos...")

try:
    embeddings = model.encode(
        texts_to_embed,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    
    print(f"Embeddings gerados! Forma: {embeddings.shape}")
    
    for idx, doc in enumerate(corpus_data):
        doc['knn_vector'] = embeddings[idx].tolist()
    
    print(f"Embeddings associados aos documentos")
except Exception as e:
    print(f"Erro ao gerar embeddings: {str(e)}")

## 8. Ingestao no OpenSearch

In [ ]:
from datetime import datetime

print("Iniciando ingestao de documentos...")

bulk_data = []
for doc in corpus_data:
    header = {
        "index": {
            "_index": INDEX_NAME,
            "_id": doc['id']
        }
    }
    
    doc_to_index = doc.copy()
    doc_to_index['timestamp'] = datetime.now().isoformat()
    
    bulk_data.append(header)
    bulk_data.append(doc_to_index)

try:
    response = client.bulk(body=bulk_data)
    
    if response['errors']:
        print(f"Alguns documentos falharam na ingestao")
    else:
        docs_ingested = len([item for item in response['items'] if 'index' in item])
        print(f"Sucesso! {docs_ingested} documentos ingeridos")
except Exception as e:
    print(f"Erro durante bulk insert: {str(e)}")

## 9. Validacao do Indice

In [ ]:
try:
    count_response = client.count(index=INDEX_NAME)
    doc_count = count_response['count']
    
    print(f"Total de documentos: {doc_count}")
    
    if doc_count == len(corpus_data):
        print(f"Correto!")
except Exception as e:
    print(f"Erro ao contar documentos: {str(e)}")

In [ ]:
# Teste kNN
query_text = "prisao preventiva requisitos"
query_embedding = model.encode(query_text, normalize_embeddings=True).tolist()

knn_query = {
    "size": 5,
    "query": {
        "knn": {
            "knn_vector": {
                "vector": query_embedding,
                "k": 5
            }
        }
    }
}

try:
    response = client.search(index=INDEX_NAME, body=knn_query)
    hits = response['hits']['hits']
    
    print(f"Busca kNN para: '{query_text}'")
    print(f"Resultados: {len(hits)}")
    for idx, hit in enumerate(hits[:3], 1):
        print(f"  {idx}. [{hit['_score']:.4f}] {hit['_source']['titulo']}")
except Exception as e:
    print(f"Erro em busca kNN: {str(e)}")

In [ ]:
# Teste BM25
bm25_query = {
    "size": 5,
    "query": {
        "match": {
            "text": {"query": "prisao preventiva"}
        }
    }
}

try:
    response = client.search(index=INDEX_NAME, body=bm25_query)
    hits = response['hits']['hits']
    
    print(f"Busca BM25 para: 'prisao preventiva'")
    print(f"Resultados: {len(hits)}")
    for idx, hit in enumerate(hits[:3], 1):
        print(f"  {idx}. [{hit['_score']:.4f}] {hit['_source']['titulo']}")
except Exception as e:
    print(f"Erro em busca BM25: {str(e)}")

## 10. Exercicio Proposto

Tarefa: Adicionar campo 'area' como keyword e reindexar documentos. Qual a vantagem de keyword vs text?

In [ ]:
print("Responda o exercicio aqui")

## Referencias ABNT

BAAI. Sentence-Transformers. Disponivel em: https://www.sbert.net/. Acesso em: 16 abr. 2024.

BRASIL. Lei no 11.340, de 7 de agosto de 2006. Diario Oficial, Brasilia, DF, 2006.

BRASIL. Lei no 13.709, de 14 de agosto de 2018. Lei Geral de Protecao de Dados. Diario Oficial, Brasilia, DF, 2018.

OPENSEARCH. OpenSearch Documentation. Disponivel em: https://opensearch.org/docs/. Acesso em: 16 abr. 2024.